In [0]:
%pip install pypdfium2==4.30.0

In [0]:
import collections
import pathlib
import re
import unicodedata

import pypdfium2 as pdfium

LANDING = pathlib.Path("/Volumes/vigia/bronze/landing/gacetas")
TABLA_INDICE = "vigia.bronze.gacetas_indice"

PROVINCIAS = ["AZUAY", "BOLIVAR", "CANAR", "CARCHI", "COTOPAXI", "CHIMBORAZO", "ELORO", "ESMERALDAS",
              "GUAYAS", "IMBABURA", "LOJA", "LOSRIOS", "MANABI", "MORONASANTIAGO", "NAPO", "PASTAZA",
              "PICHINCHA", "TUNGURAHUA", "ZAMORACHINCHIPE", "GALAPAGOS", "SUCUMBIOS", "ORELLANA",
              "SANTODOMINGO", "SANTAELENA"]
COLUMNAS = ("SE", "TOTAL", "CASOS", "TASA", "DSSA", "DCSA", "DG")
SEMANA = re.compile(r"\bSE\s*\)?\s*-?\s*(\d{1,2})\b")
CORTE = re.compile(r"(?:CORTE|ACTUALIZAD[OA])\D{0,20}(\d{1,2}/\d{1,2}/\d{2,4})")

In [0]:
def simple(texto):
    """Mayúsculas y sin tildes"""
    return unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode().upper()


def leer_paginas(ruta):
    pdf = pdfium.PdfDocument(ruta.read_bytes())
    try:
        return [simple(pdf[i].get_textpage().get_text_bounded()) for i in range(len(pdf))]
    finally:
        pdf.close()


def lineas(texto):
    return [l.strip() for l in texto.splitlines() if l.strip()]


def es_fila_de_datos(linea):
    """Las filas de la tabla empiezan con el nombre de una provincia."""
    compacta = re.sub(r"\s+", "", linea)
    return any(compacta.startswith(p) for p in PROVINCIAS)


def parece_encabezado(linea):
    """Puntaje: PROVINCIA como palabra sola y palabras de columna como SE01-38, TOTAL, CASOS o DSSA"""
    palabras = linea.split()
    return 2 * ("PROVINCIA" in palabras) + sum(p.startswith(COLUMNAS) for p in palabras)


def tabla_dengue(paginas):
    """Primera página que habla de dengue, dice PROVINCIA y nombra al menos 10 provincias
    (a comienzos de año la tabla puede traer solo las provincias con casos)."""
    for numero, texto in enumerate(paginas, start=1):
        compacto = re.sub(r"\s+", "", texto)
        cuantas = sum(p in compacto for p in PROVINCIAS)
        if "DENGUE" in texto and "PROVINCIA" in texto and cuantas >= 10:
            filas = lineas(texto)
            i = max((k for k, l in enumerate(filas) if "PROVINCIA" in l), key=lambda k: parece_encabezado(filas[k]))
            # El encabezado puede ocupar varias líneas: llega hasta la primera fila de datos
            encabezado = []
            for linea in filas[i:i + 4]:
                if es_fila_de_datos(linea):
                    break
                encabezado.append(linea)
            total = next((l for l in filas[i:] if re.match(r"TOTAL(?: GENERAL)?\s+\d", l)), "")
            return numero, cuantas, " | ".join(encabezado)[:300], total[:150]
    return None, 0, "", ""


def describir(anio, ruta):
    base = {"anio": anio, "archivo": ruta.name, "mb": round(ruta.stat().st_size / 1024**2, 2)}
    try:
        paginas = leer_paginas(ruta)
    except Exception as error:  # PDF dañado o protegido
        return {**base, "error": f"{type(error).__name__}: {error}"[:200]}
    primera = lineas(paginas[0]) if paginas else []
    titulo = next((l for l in primera if SEMANA.search(l)), primera[0] if primera else "")
    se, corte = SEMANA.search(paginas[0] if paginas else ""), CORTE.search(paginas[0] if paginas else "")
    pagina, provincias, encabezado, total = tabla_dengue(paginas)
    caracteres = sum(len(p) for p in paginas)
    prefijo = ruta.name[2:4]  # SE42__ -> 42, la semana que decía la página del MSP
    return {
        **base, "paginas": len(paginas), "caracteres": caracteres, "titulo": titulo[:160],
        "se_archivo": int(prefijo) if prefijo.isdigit() and prefijo != "00" else None,
        "se_pdf": int(se.group(1)) if se else None, "corte": corte.group(1) if corte else None,
        "pagina_tabla": pagina, "provincias_en_pagina": provincias, "encabezado": encabezado,
        "linea_total": total, "error": "sin texto: puede ser una imagen" if caracteres < 500 else None,
    }


inventario = [describir(int(c.name.split("=")[1]), r)
              for c in sorted(LANDING.glob("anio=*")) for r in sorted(c.glob("*.pdf"))]

# Gacetas con el mismo contenido: el MSP a veces enlaza el mismo PDF en dos semanas
huellas = {r.archivo: r.sha256 for r in spark.table(TABLA_INDICE).select("archivo", "sha256").collect()}
repetidas = collections.Counter(huellas.values())
for fila in inventario:
    fila["mismo_pdf_que_otra_semana"] = repetidas[huellas.get(fila["archivo"])] > 1

esquema = ("anio int, archivo string, mb double, paginas int, caracteres int, titulo string, se_archivo int, "
           "se_pdf int, corte string, pagina_tabla int, provincias_en_pagina int, encabezado string, "
           "linea_total string, error string, mismo_pdf_que_otra_semana boolean")
campos = [c.split()[0] for c in esquema.split(", ")]
display(spark.createDataFrame([[fila.get(c) for c in campos] for fila in inventario], esquema))

In [0]:
def firma(encabezado):
    """El encabezado sin números: dos gacetas con la misma firma tienen, casi seguro, el mismo formato"""
    return re.sub(r"\s+", " ", re.sub(r"\d+", "#", encabezado)).strip()[:200] or "(sin tabla de dengue)"


familias = collections.defaultdict(list)
for fila in inventario:
    familias[firma(fila.get("encabezado") or "")].append(fila)

resumen = []
for clave, filas in familias.items():
    anios = sorted({f["anio"] for f in filas})
    ejemplo = max(filas, key=lambda f: (f["anio"], f["archivo"]))
    resumen.append((clave, len(filas), f"{anios[0]}–{anios[-1]}", ejemplo["anio"], ejemplo["archivo"]))

display(spark.createDataFrame(
    sorted(resumen, key=lambda r: -r[1]),
    "familia string, gacetas int, anios string, anio_ejemplo int, archivo_ejemplo string"))